# ASR Assignment 2025-26

This notebook has been provided as a template to get you started on the assignment.  Feel free to use it for your development, or do your development directly in Python.

You can find a full description of the assignment [here](https://www.inf.ed.ac.uk/teaching/courses/asr/coursework-2026.html).

You are provided with two Python modules `observation_model.py` and `wer.py`.  The first was described in [Lab 3](https://github.com/geoph9/asr_labs/blob/main/asr_lab3_4.ipynb).  The second can be used to compute the number of substitution, deletion and insertion errors between ASR output and a reference text.

It can be used as follows:

```python
import wer

my_refence = 'A B C'
my_output = 'A C C D'

wer.compute_alignment_errors(my_reference, my_output)
```

This produces a tuple $(s,d,i)$ giving counts of substitution,
deletion and insertion errors respectively - in this example (1, 0, 1).  The function accepts either two strings, as in the example above, or two lists.  Matching is case sensitive.

## Template code

Assuming that you have already made a function to generate an WFST, `create_wfst()` and a decoder class, `MyViterbiDecoder`, you can perform recognition on all the audio files as follows:


In [1]:
import glob
import os
import wer
import observation_model
import openfst_python as fst
import math

# Create two symbol tables: one for the input labels, one for the output labels
input_sym = fst.SymbolTable()
output_sym = fst.SymbolTable()

# input symbols
input_sym.add_symbol('<eps>') # by convention, <eps> always
                              # has symbol zero

def parse_lexicon(lex_file):
    """
    Parse the lexicon file and return it in dictionary form.
    
    Args:
        lex_file (str): filename of lexicon file with structure '<word> <phone1> <phone2>...'
                        eg. peppers p eh p er z

    Returns:
        lex (dict): dictionary mapping words to list of phones
    """
    
    lex = {}  # create a dictionary for the lexicon entries (this could be a problem with larger lexica)
    with open(lex_file, 'r') as f:
        for line in f:
            line = line.split()  # split at each space
            lex[line[0]] = line[1:]  # first field the word, the rest is the phones
    return lex
    
def generate_symbol_tables(lexicon, n=3):
    '''
    Return word, phone and state symbol tables based on the supplied lexicon
    
    Args:
        lexicon (dict): lexicon to use, created from the parse_lexicon() function
        n (int): number of states for each phone HMM
        
    Returns:
        word_table (fst.SymbolTable): table of words
        phone_table (fst.SymbolTable): table of phones
        state_table (fst.SymbolTable): table of HMM phone-state IDs
    '''
    state_table = fst.SymbolTable()
    phone_table = fst.SymbolTable()
    word_table = fst.SymbolTable()
    
    # Standard epsilons
    state_table.add_symbol('<eps>') 
    phone_table.add_symbol('<eps>')
    word_table.add_symbol('<eps>')
    
    # TASK 2: Add silence symbols
    for i in range(1, 6):
        state_table.add_symbol(f"sil_{i}")
    
    for word, phones in lexicon.items():
        word_table.add_symbol(word)
        for p in phones:
            phone_table.add_symbol(p)
            for i in range(1, n+1):
                state_table.add_symbol(f"{p}_{i}")
    
    return word_table, phone_table, state_table

In [2]:
lex = parse_lexicon('lexicon.txt')
lex

{'a': ['ey'],
 'of': ['ah', 'v'],
 'peck': ['p', 'eh', 'k'],
 'peppers': ['p', 'eh', 'p', 'er', 'z'],
 'peter': ['p', 'iy', 't', 'er'],
 'picked': ['p', 'ih', 'k', 't'],
 'pickled': ['p', 'ih', 'k', 'ah', 'l', 'd'],
 'piper': ['p', 'ay', 'p', 'er'],
 'the': ['dh', 'iy'],
 "where's": ['w', 'eh', 'r', 'z']}

In [3]:
word_table, phone_table, state_table = generate_symbol_tables(lex)

In [4]:
# Checking word_table
word_table.write_text('tmp.txt')
print(open('tmp.txt').read())

<eps>	0
a	1
of	2
peck	3
peppers	4
peter	5
picked	6
pickled	7
piper	8
the	9
where's	10



In [5]:
# Checking phone_table
phone_table.write_text('tmp.txt')
print(open('tmp.txt').read())

<eps>	0
ey	1
ah	2
v	3
p	4
eh	5
k	6
er	7
z	8
iy	9
t	10
ih	11
l	12
d	13
ay	14
dh	15
w	16
r	17



In [6]:
# Checking state_table
state_table.write_text('tmp.txt')
print(open('tmp.txt').read())

<eps>	0
sil_1	1
sil_2	2
sil_3	3
sil_4	4
sil_5	5
ey_1	6
ey_2	7
ey_3	8
ah_1	9
ah_2	10
ah_3	11
v_1	12
v_2	13
v_3	14
p_1	15
p_2	16
p_3	17
eh_1	18
eh_2	19
eh_3	20
k_1	21
k_2	22
k_3	23
er_1	24
er_2	25
er_3	26
z_1	27
z_2	28
z_3	29
iy_1	30
iy_2	31
iy_3	32
t_1	33
t_2	34
t_3	35
ih_1	36
ih_2	37
ih_3	38
l_1	39
l_2	40
l_3	41
d_1	42
d_2	43
d_3	44
ay_1	45
ay_2	46
ay_3	47
dh_1	48
dh_2	49
dh_3	50
w_1	51
w_2	52
w_3	53
r_1	54
r_2	55
r_3	56



In [7]:
# NOTES: TROPICAL SEMIRING - fst.Weight("tropical", value) is the most common semiring for Viterbi decoding bc the "plus" operation is a min(), effectively picking the shortest (most probable) path
    ## by setting Pnext  = 0.9, we ensure that the probabilitiies exiting a state sum to 1.0
    ## arc labels ---- input_symbol : output_symbol / weight, where output symbol is emitted symbol and input symbol is the specific HMM state-dependent phone
    ## weight is the negative log probability or "cost" associated with taking that transition    
    
    # define probabilities, where a self-loop (staying in the same state) has Pself = 0.1, and a transition to the next-state thus has Pnext = 1 - Pself = 0.9
p_self = 0.1
p_next = 1.0 - p_self      # 0.9
p_forward = p_next / 2

# convert to negative log weights
w_self = fst.Weight("tropical", -math.log(p_self))
w_next = fst.Weight("tropical", -math.log(p_next))

In [8]:
def generate_phone_wfst_weighted(f, start_state, phone, n):
    """
    Generate a WFST representing an n-state left-to-right phone HMM.
    
    Args:
        f (fst.Fst()): an FST object, assumed to exist already
        start_state (int): the index of the first state, assumed to exist already
        phone (str): the phone label 
        n (int): number of emitting states of the HMM
        
    Returns:
        the final state of the FST
    """
    
    current_state = start_state
    
    for i in range(1, n+1):
        sym = f"{phone}_{i}"
        in_label = state_table.find(sym)
        
#         # output the phone label only on the transition out of the last state
#         if i == n:
#             out_label = phone_table.find(phone)
#         else:
        out_label = 0     # <eps>
        next_state = f.add_state()
        
        # 1. Self-loop arc: weight corresponds to p_self
        # fst.Arc(ilabel, olabel, weight, nextstate)
        f.add_arc(current_state, fst.Arc(in_label, 0, w_self, current_state))  
        
        # 2. Transition to next state: weight corresponds to p_next
        f.add_arc(current_state, fst.Arc(in_label, out_label, w_next, next_state))
        
        current_state = next_state
        
    return current_state

In [9]:
def generate_word_sequence_recognition_wfst_weighted(n, lex, word_table):
    f = fst.Fst()
    start_state = f.add_state()
    f.set_start(start_state)
    
    # Probabilities in negative log space
    num_words = len(lex)
    # Uniform branching weight: -log(1/N)
    w_branch = fst.Weight("tropical", -math.log(1.0 / num_words))
    w_zero = fst.Weight.One("tropical")
    
    for word, phones in lex.items():
        word_entry_state = f.add_state()
        
        # Get the word ID from your table
        word_id = word_table.find(word) 
        
        # ARC 1: Transition from central start to the word model
        # Input: 0 (epsilon), Output: word_id, Weight: branching cost
        f.add_arc(start_state, fst.Arc(0, word_id, w_branch, word_entry_state))
        
        # Build the phone chain
        current_state = word_entry_state
        for phone in phones:
            # IMPORTANT: Use your weighted phone generator here
            current_state = generate_phone_wfst_weighted(f, current_state, phone, n)
        
        # ARC 2: Transition back to start state to allow for sequences
        # Input: 0, Output: 0, Weight: 0 (no penalty for adding a word)
        f.add_arc(current_state, fst.Arc(0, 0, w_zero, start_state))
    
    # Set the start state as final so the sequence can end after any word
    f.set_final(start_state, w_zero)
    f.set_input_symbols(state_table)
    f.set_output_symbols(word_table)
    
    return f

In [10]:
class MyViterbiDecoder:
    
    NLL_ZERO = 1e10  # define a constant representing -log(0).  This is really infinite, but approximate
                     # it here with a very large number
    
    def __init__(self, f, audio_file_name, om=None):
        """Set up the decoder class with an audio file and WFST f
        """
        self.om = om if om is not None else observation_model.ObservationModel()
        self.f = f
        
        if audio_file_name:
            self.om.load_audio(audio_file_name)
        else:
            self.om.load_dummy_audio()
        
        self.initialise_decoding()

        
    def initialise_decoding(self):
        """set up the values for V_j(0) (as negative log-likelihoods)
        
        """
        
        self.V = []   # stores likelihood along best path reaching state j
        self.B = []   # stores identity of best previous state reaching state j
        self.W = []   # stores output labels sequence along arc reaching j - this removes need for 
                      # extra code to read the output sequence along the best path
        
        for t in range(self.om.observation_length()+1):
            self.V.append([self.NLL_ZERO]*self.f.num_states()) #  multiplying the empty list doesn't make multiple
            self.B.append([-1]*self.f.num_states()) # initialise backpointers with -1
            self.W.append([[] for i in range(self.f.num_states())]) # initialise word labels as empty lists
        
        # The above code means that self.V[t][j] for t = 0, ... T gives the Viterbi cost
        # of state j, time t (in negative log-likelihood form)
        # Initialising the costs to NLL_ZERO effectively means zero probability    
        
        # give the WFST start state a probability of 1.0   (NLL = 0.0)
        self.V[0][self.f.start()] = 0.0
        
        # some WFSTs might have arcs with epsilon on the input (you might have already created 
        # examples of these in earlier labs) these correspond to non-emitting states, 
        # which means that we need to process them without stepping forward in time.  
        # Don't worry too much about this!  
        self.traverse_epsilon_arcs(0)        
        
    def traverse_epsilon_arcs(self, t):
        """Traverse arcs with <eps> on the input at time t
        
        These correspond to transitions that don't emit an observation
        
        We've implemented this function for you as it's slightly trickier than
        the normal case.  You might like to look at it to see what's going on, but
        don't worry if you can't fully follow it.
        
        """
        
        states_to_traverse = list(self.f.states()) # traverse all states
        while states_to_traverse:
            
            # Set i to the ID of the current state, the first 
            # item in the list (and remove it from the list)
            i = states_to_traverse.pop(0)   
        
            # don't bother traversing states which have zero probability
            if self.V[t][i] == self.NLL_ZERO:
                    continue
        
            for arc in self.f.arcs(i):
                
                if arc.ilabel == 0:     # if <eps> transition
                  
                    j = arc.nextstate   # ID of next state  
                
                    if self.V[t][j] > self.V[t][i] + float(arc.weight):
                        
                        # this means we've found a lower-cost path to
                        # state j at time t.  We might need to add it
                        # back to the processing queue.
                        self.V[t][j] = self.V[t][i] + float(arc.weight)
                        
                        # save backtrace information.  In the case of an epsilon transition, 
                        # we save the identity of the best state at t-1.  This means we may not
                        # be able to fully recover the best path, but to do otherwise would
                        # require a more complicated way of storing backtrace information
                        self.B[t][j] = self.B[t][i] 
                        
                        # and save the output labels encountered - this is a list, because
                        # there could be multiple output labels (in the case of <eps> arcs)
                        if arc.olabel != 0:
                            self.W[t][j] = self.W[t][i] + [arc.olabel]
                        else:
                            self.W[t][j] = self.W[t][i]
                        
                        if j not in states_to_traverse:
                            states_to_traverse.append(j)

    
    def forward_step(self, t):
          
        for i in self.f.states():
            
            if not self.V[t-1][i] == self.NLL_ZERO:   # no point in propagating states with zero probability
                
                for arc in self.f.arcs(i):
                    
                    if arc.ilabel != 0: # <eps> transitions don't emit an observation
                        j = arc.nextstate
                        tp = float(arc.weight)  # transition prob
                        ep = -self.om.log_observation_probability(self.f.input_symbols().find(arc.ilabel), t)  # emission negative log prob
                        prob = tp + ep + self.V[t-1][i] # they're logs
                        if prob < self.V[t][j]:
                            self.V[t][j] = prob
                            self.B[t][j] = i
                            
                            # store the output labels encountered too
                            if arc.olabel !=0:
                                self.W[t][j] = [arc.olabel]
                            else:
                                self.W[t][j] = []
                            
    
    def finalise_decoding(self):
        """ this incorporates the probability of terminating at each state
        """
        
        for state in self.f.states():
            final_weight = float(self.f.final(state))
            if self.V[-1][state] != self.NLL_ZERO:
                if final_weight == math.inf:
                    self.V[-1][state] = self.NLL_ZERO  # effectively says that we can't end in this state
                else:
                    self.V[-1][state] += final_weight
                    
        # get a list of all states where there was a path ending with non-zero probability
        finished = [x for x in self.V[-1] if x < self.NLL_ZERO]
        if not finished:  # if empty
            print("No path got to the end of the observations.")
        
        
    def decode(self):
        self.initialise_decoding()
        t = 1
        while t <= self.om.observation_length():
            self.forward_step(t)
            self.traverse_epsilon_arcs(t)
            t += 1
        self.finalise_decoding()
    
    def backtrace(self):
        # T is the total number of frames (observations)
        T = self.om.observation_length()

        # 1. Find the best final state at time T
        # We look for the state in the last column of V with the lowest cost
        best_final_state = -1
        min_cost = self.NLL_ZERO

        for s in range(self.f.num_states()):
            if self.V[T][s] < min_cost:
                min_cost = self.V[T][s]
                best_final_state = s

        # If no path reached the end, return empty
        if best_final_state == -1:
            print("Backtrace failed: No valid path found.")
            return [], ""

        # 2. Trace back from time T to time 1
        state_seq = [best_final_state]
        label_seq = []
        curr_state = best_final_state

        for t in range(T, 0, -1):
            # W[t][curr_state] contains the output labels (words/phones) 
            # that were emitted to reach this state at this time step.
            # We prepend them because we are moving backward.
            label_seq = self.W[t][curr_state] + label_seq

            # Follow the back-pointer to the state at time t-1
            curr_state = self.B[t][curr_state]
            state_seq.append(curr_state)

        # 3. Finalize the sequences
        # Reverse the state sequence to go from start -> end
        state_seq.reverse()

        # Map integer label IDs back to human-readable strings using the Symbol Table
        symbol_table = self.f.output_symbols()
        # Filter out epsilons (label 0) and join labels with a space
        words = " ".join([symbol_table.find(l) for l in label_seq if l != 0])

        return state_seq, words

In [11]:
decoder = MyViterbiDecoder(generate_word_sequence_recognition_wfst_weighted(3, lex, word_table), '')   # empty string '' just means use dummy probabilities for testing
decoder.decode()

In [12]:
help(decoder)

Help on MyViterbiDecoder in module __main__ object:

class MyViterbiDecoder(builtins.object)
 |  MyViterbiDecoder(f, audio_file_name, om=None)
 |  
 |  Methods defined here:
 |  
 |  __init__(self, f, audio_file_name, om=None)
 |      Set up the decoder class with an audio file and WFST f
 |  
 |  backtrace(self)
 |  
 |  decode(self)
 |  
 |  finalise_decoding(self)
 |      this incorporates the probability of terminating at each state
 |  
 |  forward_step(self, t)
 |  
 |  initialise_decoding(self)
 |      set up the values for V_j(0) (as negative log-likelihoods)
 |  
 |  traverse_epsilon_arcs(self, t)
 |      Traverse arcs with <eps> on the input at time t
 |      
 |      These correspond to transitions that don't emit an observation
 |      
 |      We've implemented this function for you as it's slightly trickier than
 |      the normal case.  You might like to look at it to see what's going on, but
 |      don't worry if you can't fully follow it.
 |  
 |  --------------------

In [13]:
def read_transcription(wav_file):
    """
    Get the transcription corresponding to wav_file.
    """
    
    transcription_file = os.path.splitext(wav_file)[0] + '.txt'
    
    with open(transcription_file, 'r') as f:
        transcription = f.readline().strip()
    
    return transcription

In [23]:
## Task 1 - this is complete and works
## Note to Joyce
### See the 2 word documents for the output and the general "analysis" of the output after task 1

In [15]:
# CODE FOR TASK 1 DO NOT REMOVE

# # 1. Setup
# lex = parse_lexicon('lexicon.txt')
# word_table, phone_table, state_table = generate_symbol_tables(lex)
# recognition_wfst = generate_word_sequence_recognition_wfst_weighted(3, lex, word_table)

# # 2. Batch Processing
# total_s, total_d, total_i, total_n = 0, 0, 0, 0

# for wav_file in glob.glob('/group/teaching/asr/labs/recordings/*.wav'):
#     # Initialize decoder for this specific file
#     decoder = MyViterbiDecoder(recognition_wfst, wav_file)
#     decoder.decode()
    
#     # Get the recognized words (hypothesis)
#     state_path, hypothesis = decoder.backtrace()
    
#     # Get the actual transcription (reference)
#     reference = read_transcription(wav_file)
    
#     # Calculate errors
#     s, d, i = wer.compute_alignment_errors(reference, hypothesis)
    
#     # Accumulate for overall WER
#     total_s += s
#     total_d += d
#     total_i += i
#     total_n += len(reference.split())
    
#     print(f"File: {os.path.basename(wav_file)}")
#     print(f"  Ref: {reference}")
#     print(f"  Hyp: {hypothesis}")
#     print(f"  Errors: S:{s} D:{d} I:{i}")

# # 3. Final Analysis
# overall_wer = (total_s + total_d + total_i) / total_n

# print(f"\nTOTAL WORD COUNT: {total_count}")
# print(f"\nERROR COUNTS: S - {total_s}, D - {total_d}, I - {total_i}")
# print(f"\nFINAL WER: {overall_wer * 100:.2f}%")

In [16]:
## Task 2 - in progress

In [17]:
# 1. Count unigrams from your training transcripts
num_files = 0
unigram_counts = {}
lex_words = lex.keys()   

for txt_file_path in glob.glob('/group/teaching/asr/labs/recordings/*.txt'):
    num_files += 1
    
    with open(txt_file_path, 'r') as f:
        for line in f:
            for word in line.split():
                if word in lex_words:
                    unigram_counts[word] = unigram_counts.get(word, 0) + 1

total_count = sum(unigram_counts.values())
print(f"Processed {num_files} files. Total words counted: {total_count}")
print(unigram_counts)

Processed 216 files. Total words counted: 1534
{'peter': 182, 'piper': 178, 'picked': 182, 'the': 101, 'peck': 180, 'of': 164, 'peppers': 190, "where's": 84, 'a': 101, 'pickled': 172}


In [18]:
def generate_tuned_recognition_wfst(n, lex, word_table, state_table, unigram_counts, total_count, wip):
    f = fst.Fst()
    hub = f.add_state()
    f.set_start(hub)
    
    # --- SILENCE MODEL ---
    sil_states = [f.add_state() for _ in range(5)]
    w_move = fst.Weight("tropical", -math.log(0.5))
    
    # Hub to Sil State 1
    f.add_arc(hub, fst.Arc(0, 0, fst.Weight.One("tropical"), sil_states[0]))
    
    # State 1 -> 2, 3, 4 (Label: sil 1)
    for next_idx in [1, 2, 3]:
        f.add_arc(sil_states[0], fst.Arc(state_table.find("sil_1"), 0, w_move, sil_states[next_idx]))
        
    # States 2, 3, 4 (Ergodic - can loop to 2, 3, 4 or move to 5)
    for i in [1, 2, 3]: 
        label = state_table.find(f"sil_{i+1}")
        for j in [1, 2, 3, 4]:
            f.add_arc(sil_states[i], fst.Arc(label, 0, w_move, sil_states[j]))
            
    # State 5 -> Hub
    f.add_arc(sil_states[4], fst.Arc(state_table.find("sil_5"), 0, fst.Weight.One("tropical"), hub))

    # --- WORD LOOP ---
    for word, phones in lex.items():
        prob = unigram_counts.get(word, 1) / total_count
        w_entry = fst.Weight("tropical", -math.log(prob) + wip)
        
        entry_s = f.add_state()
        f.add_arc(hub, fst.Arc(0, word_table.find(word), w_entry, entry_s))
        
        curr = entry_s
        for p in phones:
            curr = generate_phone_wfst_weighted(f, curr, p, n)
            
        f.add_arc(curr, fst.Arc(0, 0, fst.Weight.One("tropical"), hub))
    
    f.set_final(hub, fst.Weight.One("tropical"))
    f.set_input_symbols(state_table)
    f.set_output_symbols(word_table)
    return f

In [19]:
## Note to Joyce
### This code runs with parameters WIP = 12.0 and P_SELF = 0.1. WIP can and should be tweaked. See the below function "evaluate_wip" for my attempt to do this efficiently.
### P_self can (and should) ALSO be tweaked. I think a similar approach to what I have attempted in "evaluate_wip" would be good. Suggested P_self = 0.05, 0.1 (default), 0.15

In [20]:
# Task 2 Parameters to experiment with:
## WIP = 12.0      # Word Insertion Penalty: increase this to reduce 'the the the'
## P_SELF = 0.1    # Self-loop probability for phones

# shared_om = observation_model.ObservationModel()

# # Initialize overall recognition WFST
# f_tuned = generate_tuned_recognition_wfst(3, lex, word_table, state_table, unigram_counts, total_count, WIP)

# total_s, total_d, total_i, total_n = 0, 0, 0, 0

# for wav_file in glob.glob('/group/teaching/asr/labs/recordings/*.wav'):
#     try:
#         decoder = MyViterbiDecoder(f_tuned, wav_file, om=shared_om)
#         decoder.decode()
        
#         _, hypothesis = decoder.backtrace()
#         reference = read_transcription(wav_file)
        
#         s, d, i = wer.compute_alignment_errors(reference, hypothesis)
        
#         # Proper syntax for increments
#         total_s += s
#         total_d += d
#         total_i += i
#         total_n += len(reference.split())

#         print(f"File: {os.path.basename(wav_file)}")
#         print(f"  Ref: {reference}")
#         print(f"  Hyp: {hypothesis}")
#         print(f"  Errors: S:{s} D:{d} I:{i}")

#     except KeyError as e:
#         print(f"Error in {os.path.basename(wav_file)}: Label {e} not found in ObservationModel state_map.")
#         print("Check if you should use 'sil 1' (space) or 'sil_1' (underscore).")
#         break # Stop early so you can fix the naming convention

# if total_n > 0:
#     print(f"\nTOTAL WORD COUNT: {total_count}")
#     print(f"\nERROR COUNTS: S - {total_s}, D - {total_d}, I - {total_i}")
#     print(f"\nFinal WER: {((total_s + total_d + total_i) / total_n) * 100:.2f}%")



In [21]:
## Note to Joyce
### I am attempting to automate the allocation of WIP (in integers) until the system is tuned (WER is minimised)
### I don't know if this code runs as it should (it was taking ages), but ideally, it will highlight the correct/most optimal WIP to minimise the WER

In [22]:
def evaluate_wip(wip_value, lex, word_table, state_table, unigram_counts, total_count, wav_files):
    # 1. Generate WFST with the specific WIP
    f_tuned = generate_tuned_recognition_wfst(3, lex, word_table, state_table, unigram_counts, total_count, wip_value)
    
    total_s, total_d, total_i, total_n = 0, 0, 0, 0
    shared_om = observation_model.ObservationModel()

    # 2. Process all files in the test set
    for wav_file in wav_files:
        decoder = MyViterbiDecoder(f_tuned, wav_file, om=shared_om)
        decoder.decode()
        _, hypothesis = decoder.backtrace()
        reference = read_transcription(wav_file)
        
        s, d, i = wer.compute_alignment_errors(reference, hypothesis)
        total_s += s
        total_d += d
        total_i += i
        total_n += len(reference.split())
    
    wer_score = (total_s + total_d + total_i) / total_n
    return wer_score, (total_s, total_d, total_i)

# --- Optimization Execution ---

# Define the range of WIP values you want to test
# didn't include 0 as that would have no effect
wip_range = [5.0, 10.0, 12.0, 15.0, 20.0, 25.0]
results = []
wav_files = glob.glob('/group/teaching/asr/labs/recordings/*.wav')

print(f"{'WIP':>6} | {'WER (%)':>10} | {'S':>4} {'D':>4} {'I':>4}")
print("-" * 35)

for val in wip_range:
    error_rate, counts = evaluate_wip(val, lex, word_table, state_table, unigram_counts, total_count, wav_files)
    results.append((val, error_rate))
    s, d, i = counts
    print(f"{val:6.1f} | {error_rate*100:10.2f}% | {s:4} {d:4} {i:4}")

# Find the best WIP
best_wip, min_wer = min(results, key=lambda x: x[1])
print(f"\nOptimal WIP: {best_wip} with WER: {min_wer*100:.2f}%")

   WIP |    WER (%) |    S    D    I
-----------------------------------


KeyboardInterrupt: 

In [ ]:
## Note to Joyce
### This is an attempt to add optional silence as in the assignment doc it says:
""" Investigate the effect of adding an optional silence state between words in your WFST
to handle the case where there are pauses between words in the recording. The provided
observation model can compute the observation probability for silence using the label
“sil N” for N = (1, 2, . . . 5). (This has been trained using a five-state left-to-right
topology where states 1 and 5 are initial and final states respectively, and states 2, 3
and 4 have an ergodic structure. You do not need to replicate this topology exactly)"""
### Haven't yet had a chance to implement/test it fully (it's not used anywhere in the code), so feel free to have a crack at it :D

In [ ]:
def add_optional_silence(f, central_state, state_table):
    """
    Adds a silence HMM to the hub of the WFST.
    """
    # 1. Create the 5 states for the silence model
    sil_states = [f.add_state() for _ in range(5)]
    w_move = fst.Weight("tropical", -math.log(0.5)) # Equal probability to stay/move
    
    # 2. Entrance from Hub to Sil State 1 (Epsilon transition)
    f.add_arc(central_state, fst.Arc(0, 0, fst.Weight.One("tropical"), sil_states[0]))
    
    # 3. Transitions between states (using "sil_N" labels)
    # State 1 (Initial) -> States 2, 3, 4
    for next_s in [1, 2, 3]:
        label = state_table.find(f"sil_1") 
        f.add_arc(sil_states[0], fst.Arc(label, 0, w_move, sil_states[next_s]))

    # States 2, 3, 4 (Ergodic structure)
    for i in [1, 2, 3]:
        label = state_table.find(f"sil_{i+1}")
        for j in [1, 2, 3, 4]: # Can stay in 2,3,4 or move to 5
            f.add_arc(sil_states[i], fst.Arc(label, 0, w_move, sil_states[j]))

    # 4. Exit from Sil State 5 back to Hub
    f.add_arc(sil_states[4], fst.Arc(0, 0, fst.Weight.One("tropical"), central_state))